# Fashion Product Retrieval: CLIP + Trained Residual Projection Heads

**Authors:** Hiren Waghela and Khushi Parekh  
**Proposed topic:** Contrastive Learning-Based Multimodal Fashion Product Retrieval Using Vision–Language Transformers

This is the complete, self-contained project notebook. No companion Python files
are needed to run it. It includes the project specification, training, baseline
comparisons, evaluation, plots, retrieval demo, and deployment export.

**Before starting:** In Colab choose **Runtime → Change runtime type → GPU**,
then run cells in order. The first cell installs dependencies. The next setup
cell mounts Drive for persistent outputs. Dataset download is public where
Kaggle permits; if authentication is requested, authenticate with KaggleHub or
set `DATASET_DIR` to a dataset folder already available in Colab/Drive.

For a quick check, change `MAX_PRODUCTS=2000`, `EPOCHS=2`, and `TRAIN_SEEDS=[42]`.
Use all products and all three seeds for the final experiment. No results are
pre-filled and improvement is not guaranteed. Colab GPU availability varies.


# Contrastive Learning-Based Multimodal Fashion Product Retrieval Using Vision–Language Transformers

## Project description

This project studies whether lightweight domain adaptation improves fashion
product retrieval over a frozen, general-purpose vision–language model. It uses
OpenAI CLIP ViT-B/32 to extract image and text features, trains separate residual
neural projection heads using a symmetric contrastive objective, and retrieves
products by cosine similarity. The deliverables are a trained adapter model,
reproducible baseline comparisons, saved evaluation results, a retrieval demo,
and FAISS indexes for later application integration.

The research question is: **Do trained residual projection heads improve
held-out fashion image–text retrieval relative to original CLIP?** Improvement
is a hypothesis, not a promised outcome. Report decreases or no improvement
with the same care as increases.

The original CLIP encoders stay frozen. This is training a domain adaptation
model on pretrained features, not training CLIP from scratch and not fine-tuning
the Transformer encoders. No generative LLM or CNN is required.

## Dataset and inputs

- Dataset: Fashion Product Images (Small), by Param Aggarwal.
- Dataset link: https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small
- KaggleHub handle: `paramaggarwal/fashion-product-images-small`.
- Use `styles.csv` or an existing `cleaned_products.csv` and the product images.
- These images are small thumbnails; resizing to 224 pixels does not recover
  missing visual detail. Use the same source images for every comparison.
- Construct descriptions deterministically, without an LLM, from product name,
  article type, colour, audience, master category, and usage.
- Example: `Puma Men Grey T-shirt. Type: Tshirts. Color: Grey. Audience: Men.
  Category: Apparel. Usage: Casual.`
- Skip malformed CSV records with a reported count, missing images, corrupt
  images, duplicate product IDs, and empty descriptions. Save an audit CSV.
- Treat paths in an old CSV as nonportable: resolve images by product ID.
- RGB images use the checkpoint's CLIP processor: bicubic resize, center crop
  to 224 x 224, rescaling and checkpoint mean/std normalization.
- Text uses the checkpoint tokenizer with padding, truncation, and at most
  77 tokens, including special tokens. Report the number truncated.

## Split and duplicate policy

Use a fixed group-based 70/15/15 train/validation/test split with split seed 42.
The percentages are approximate because a duplicate group must remain intact.
Create connected components by linking equal normalized descriptions OR equal
decoded RGB pixel hashes (including dimensions). All members of a connected
component belong to one split. Assert no component overlaps splits.

Hashing decoded pixels catches identical images encoded differently, not every
visually similar image. Near-duplicate leakage remains a limitation. Grouping
by descriptions can join distinct but identically described products; explain
this evaluation convention. Save product IDs, captions, image hashes,
duplicate-group IDs, and split membership before training.

The paired metadata descriptions are structured captions, not independently
annotated human search queries. Performance on arbitrary short user queries
must not be inferred from paired-caption scores alone.

## Proposed architecture

```
RGB image -> frozen CLIP ViT-B/32 -> normalized 512-D feature
                                       |
                         trainable image residual head
                                       |
                              normalized 512-D vector

Description -> frozen CLIP text Transformer -> normalized 512-D feature
                                                 |
                                  trainable text residual head
                                                 |
                                        normalized 512-D vector
```

CLIP ViT-B/32 has a 12-layer vision Transformer (hidden width 768, 12 heads,
32 x 32 patches) and a 12-layer text Transformer (hidden width 512, 8 heads).
Both output 512-dimensional projected features. Load
`openai/clip-vit-base-patch32`; never initialize random encoder weights.

Each independent residual head is:

`Linear(512, 256) -> GELU -> Dropout(0.1) -> Linear(256, 512)`

For normalized frozen feature x, output:

`z = L2Normalize(x + 0.2 * MLP(x))`

Zero-initialize the final linear layer so that the initial representation is
exactly the original normalized CLIP representation. The residual weight 0.2
is fixed. Each head has 262,912 parameters; both have 525,824. A learned scalar
logit scale adds one, for **525,825 trainable parameters**. Print measured
parameter counts for the downloaded CLIP checkpoint and adapter model.

## Training objective

For a paired batch of B products, compute:

`logits[i,j] = exp(logit_scale) * dot(image_z[i], text_z[j])`

Initialize temperature to 0.07 (`logit_scale = log(1/0.07)`) and clamp the
scale between 1 and 100 after optimization. Pairs sharing a duplicate-group
ID are positives. Other batch entries are negatives. Use a row-normalized
uniform target over all positives and cross-entropy against log-softmax,
in both image-to-text and text-to-image directions. Average the two losses.
This is a multi-positive symmetric contrastive objective; do not treat equal
captions as independent negatives.

Cache the frozen, normalized embeddings once in float32. The heads train on
these cached features. No pixel augmentation occurs during head training.
Adding augmentation would require extracting additional frozen views and a
separate controlled experiment. Ordinary gradient accumulation does not
increase the number of in-batch negatives; do not claim that it does.

## Default parameters

| Parameter | Default |
|---|---|
| Encoder | `openai/clip-vit-base-patch32` |
| Encoder updates | None; frozen and evaluation mode |
| CLIP extraction batch | 64; lower to 32/16 if GPU memory is insufficient |
| CLIP extraction precision | CUDA float16 autocast; CPU float32 |
| Saved feature precision | float32, L2 normalized |
| Training precision | float32; small heads do not need AMP |
| Training batch size | 256 |
| Maximum epochs | 20 |
| Optimizer | AdamW |
| Head learning rate | 0.001 |
| Weight decay | 0.0001 on head weights; zero on biases and logit scale |
| Adam betas | (0.9, 0.999) |
| Schedule | One epoch linear warmup, then cosine decay |
| Minimum learning-rate multiplier | 0.01 |
| Gradient clipping | Global norm 1.0 |
| Hidden width / output width | 256 / 512 |
| Dropout / residual weight | 0.1 / 0.2 |
| Initial temperature | 0.07, learned thereafter |
| Early-stop patience | 5 epochs |
| Minimum selection improvement | 0.0001 |
| Selection criterion | Mean validation bidirectional MRR@10 |
| Training seeds | 42, 123, 2026 |
| Split seed | 42, fixed across training seeds |
| Feature DataLoader workers | 0; cached tensors are already in memory |
| Retrieval evaluation query block | 256 |
| Retrieval cutoffs | 1, 5, 10 |

For a quick notebook check, set `MAX_PRODUCTS=2000`, `EPOCHS=2`, and
`TRAIN_SEEDS=[42]`. Such a run is a smoke run, not the final experiment.
For final results, leave `MAX_PRODUCTS=None` and use the default three seeds.
Do not prescribe a training time before measuring the actual Colab runtime.

Save output to Google Drive by default, while downloading the dataset to the
local Colab runtime for faster image reads. Free Colab GPU availability and
session duration are variable. CPU fallback should work but extraction can be
much slower. Checkpoint each epoch and resume interrupted training with model,
optimizer, scheduler, epoch, best selection score, history, and RNG state.

## Evaluation and comparisons

Evaluate ORIGINAL CLIP versus TRAINED RESIDUAL HEADS on exactly the same splits,
descriptions, galleries, and relevance labels. Validation controls checkpoint
selection; do not evaluate or tune on test data until training is complete.
Use the validation-best checkpoint for each training seed. Select the deployable
seed using validation only, never by its test score.

Primary tasks:

1. Text-to-image retrieval: held-out captions query held-out product images.
2. Image-to-text retrieval: held-out images query held-out product captions.

Relevance means the query and gallery item share the saved duplicate-group ID.
The test gallery contains only test products and its size must be reported.
The paired target belongs in this cross-modal gallery. This is not an
image-self-retrieval experiment.

Report:

- **Success@1/5/10**: fraction of queries with at least one relevant result.
  This is often called Recall@K in single-target retrieval papers; use the
  explicit Success label here because some queries have multiple positives.
- **Recall@1/5/10**: fraction of all relevant gallery items retrieved, averaged
  over queries. Do not silently conflate this with Success@K.
- **MRR@10**: reciprocal rank of the first relevant result within top ten,
  or zero when none is present.
- **nDCG@10**: binary relevance, normalized by the ideal top-ten ranking.
- Mean and sample standard deviation across the three training seeds for the
  adapted model; baseline is deterministic for the cached embeddings.
- Training loss, validation selection score, learning rate and temperature
  curves; dataset distributions and parameter counts.
- Latency: exact/FAISS search-only time separately from text encoding, adapter
  transformation, and search end-to-end time. Use warmup and repeated timings,
  explicitly excluding model loading and image display.

Exact retrieval must be blocked rather than materializing an entire
N x N similarity matrix. Use stable catalog-row ordering to break ties.

Secondary, optional ranking experiment: deterministic short queries made from
article type, colour, and audience. Relevance is exact agreement on those
attributes, so report the experiment as synthetic metadata-query retrieval.
Compare original CLIP, adapted embeddings, and adapted embeddings plus a
fixed attribute-based reranker. Keep relevance separate from pair-group
relevance, disclose that the reranker accesses catalog metadata, and do not
interpret this as a human-query benchmark. Use one query per unique attribute
tuple, skip missing attributes, and report the number of queries/gallery items.
This optional experiment should not control checkpoint selection.

## Demo and export

- Text query searches the IMAGE index directly using adapted text features.
- Uploaded image searches the IMAGE index using adapted image features.
- A combined query uses the normalized weighted sum of adapted text and image
  vectors, with text weight alpha=0.5 by default. This is an inference heuristic,
  not a separately trained fusion module. Display top ten product thumbnails.
- Exclude the source product itself when demonstrating catalog-image search.
- Use FAISS `IndexFlatIP` on normalized float32 vectors. It is exact search;
  FAISS handles retrieval, not deep learning.
- Export `fashion_adapters.pt`, `fashion_adapters.py`, `model_config.json`,
  `catalog_metadata.csv`, `image_index.faiss`, `text_index.faiss`, adapted
  embeddings, split manifest, cleaning audit, checkpoint histories, raw metric
  CSV/JSON files, summary tables, PNG plots, requirements freeze, and a README.
- Checkpoint carries config, base model ID/revision, manifest fingerprint,
  state_dict, training seed, best epoch and validation score.
- Export JSON/CSV values as native Python numbers, not unserializable tensors.
- Bundle deployment files in a ZIP; keep large cached baseline features and
  resumable checkpoints in Drive rather than duplicating them in the ZIP.

## Existing application integration contract

The notebook does not modify the running application. After training:

1. Load the exported adapter definitions and checkpoint alongside the SAME
   base CLIP checkpoint and processor.
2. Apply `text_adapter` after CLIP text embedding and `image_adapter` after
   CLIP image embedding. Normalize their outputs.
3. Load the newly exported indexes, never old indexes with new query vectors.
4. Every index row maps to the SAME row in `catalog_metadata.csv`. Resolve
   thumbnail paths locally by product ID; exported Colab paths are nonportable.
5. For the proposed text-to-image system, route text search to
   `image_index.faiss`. `text_index.faiss` is for image-to-text experiments or
   a separate text-to-text search, not the primary text-to-image search.
6. Existing category/audience filters can still constrain catalog candidates.
   Describe additional score boosts separately from the trained model.

## Limitations and report claims

This is a feasible student experiment in lightweight domain adaptation, not a
claim of a novel architecture or state-of-the-art performance. It trains small
neural heads on learned Transformer features, rather than adapting the encoder
weights. Synthetic descriptions, thumbnail quality, annotation noise,
unidentified near duplicates, and ambiguous captions limit interpretation.
The matched-pair metric does not fully measure open-ended shopping relevance.
Potential extensions include partially unfreezing the final Transformer block,
LoRA, hard-negative selection, human-labeled queries, and larger images; each
requires a separate experiment and changed compute budget.

## Copyable prompt for GPT

Generate a complete, executable Google Colab .ipynb implementing this
specification. Include clear Markdown explanations before each stage and
code cells with no TODO placeholders. Preserve the frozen-encoder/residual-head
scope and the exact evaluation definitions. Use KaggleHub or a user-configured
dataset directory; support Google Drive output, feature caching with manifest
and checkpoint revision validation, resumable training, CPU fallback, three
training seeds, baseline comparisons, honest held-out evaluation, an interactive
retrieval demo, figures, and a deployment ZIP. Include internal correctness
checks for parameter counts, unit-length features, duplicate split isolation,
multi-positive loss, identity initialization, checkpoint loading, and row-order
alignment. Do not fabricate results, promise improvement, or include test data
in training/selection. The delivered notebook must run in order after the user
selects a Colab GPU runtime and supplies any required dataset authentication.

## Primary references

- CLIP paper: https://arxiv.org/abs/2103.00020
- CLIP model/processor API: https://huggingface.co/docs/transformers/model_doc/clip
- KaggleHub dataset download: https://github.com/Kaggle/kagglehub
- PyTorch AdamW: https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html
- PyTorch mixed precision: https://docs.pytorch.org/docs/stable/notes/amp_examples.html


## 1. Install dependencies

Keep Colab's preinstalled PyTorch/CUDA combination. Use Transformers 4.x to
match the existing application's CLIP API. Save exact resolved package versions
in the outputs for reproducibility. Installation may request a runtime restart;
if so, restart and continue from this cell.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.45,<5", "kagglehub>=0.3,<1",
    "pandas>=2,<3", "numpy>=1.26,<3", "Pillow>=10,<13",
    "matplotlib>=3.8,<4", "tqdm>=4.66,<5", "faiss-cpu>=1.8,<2",
])


## 2. Configuration and runtime

All experiment settings are in one place. Extraction and head-training batches
are different: CLIP runs on raw images once, while training uses cached features.
CPU execution works, although CLIP extraction is considerably slower.


In [ ]:
import csv
import hashlib
import io
import json
import math
import os
import random
import re
import shutil
import time
import warnings
import zipfile
from collections import Counter
from contextlib import nullcontext
from pathlib import Path

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor

MODEL_ID = "openai/clip-vit-base-patch32"
MODEL_REVISION = None  # resolved HF commit is recorded before feature extraction
DATASET_HANDLE = "paramaggarwal/fashion-product-images-small"
DATASET_DIR = None  # e.g. "/content/drive/MyDrive/fashion-product-images-small"
SAVE_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/fashion_clip_experiment"
LOCAL_OUTPUT_DIR = "/content/fashion_clip_experiment"
MAX_PRODUCTS = None
SPLIT_SEED = 42
TRAIN_SEEDS = [42, 123, 2026]
EXTRACT_BATCH_SIZE = 64
TRAIN_BATCH_SIZE = 256
EVAL_BATCH_SIZE = 256
EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 1
MIN_LR_MULTIPLIER = 0.01
PATIENCE = 5
MIN_DELTA = 1e-4
GRAD_CLIP_NORM = 1.0
HIDDEN_DIM = 256
DROPOUT = 0.1
RESIDUAL_WEIGHT = 0.2
INITIAL_TEMPERATURE = 0.07
RESUME_TRAINING = True
RUN_METADATA_QUERY_EXPERIMENT = True
METADATA_QUERY_LIMIT = 300
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert TRAIN_SEEDS and len(set(TRAIN_SEEDS)) == len(TRAIN_SEEDS)
assert EPOCHS >= 1 and TRAIN_BATCH_SIZE >= 2 and EXTRACT_BATCH_SIZE >= 1
assert MAX_PRODUCTS is None or MAX_PRODUCTS >= 30
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Outside Colab, set SAVE_TO_DRIVE=False.") from exc
    drive.mount("/content/drive")
OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR if SAVE_TO_DRIVE else LOCAL_OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE, "| PyTorch:", torch.__version__)
if DEVICE.type == "cuda":
    info = torch.cuda.get_device_properties(0)
    print("GPU:", info.name, "| VRAM GB:", round(info.total_memory / 2**30, 2))
else:
    print("CPU fallback: enable a GPU runtime for faster feature extraction.")
print("Persistent output folder:", OUTPUT_DIR)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SPLIT_SEED)
with (OUTPUT_DIR / "requirements_freeze.txt").open("w") as requirements_file:
    subprocess.check_call([sys.executable, "-m", "pip", "freeze"], stdout=requirements_file)


## 3. Download data, clean records, and group duplicates

Resolve thumbnails by product ID, validate decoded RGB pixels, and log every
discarded row. Identical descriptions and identical decoded images form connected
duplicate groups. Grouping happens before splitting, avoiding duplicate groups
appearing in both training and evaluation.


In [ ]:
def normalize_caption(text):
    return re.sub(r"\s+", " ", str(text).strip().casefold())

def build_caption(row):
    name = str(row.get("productDisplayName", "")).strip()
    fields = [("Type", "articleType"), ("Color", "baseColour"),
              ("Audience", "gender"), ("Category", "masterCategory"), ("Usage", "usage")]
    parts = [name] if name else []
    parts.extend(f"{label}: {str(row.get(key, '')).strip()}" for label, key in fields
                 if str(row.get(key, "")).strip())
    return ". ".join(parts) + "." if parts else ""

def decoded_image_hash(path):
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
        digest = hashlib.sha256()
        digest.update(f"{image.width}x{image.height}:RGB:".encode())
        digest.update(image.tobytes())
        return digest.hexdigest()

def duplicate_groups(frame):
    parent = list(range(len(frame)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    def union(a, b):
        a, b = find(a), find(b)
        if a != b:
            parent[max(a, b)] = min(a, b)
    for column in ("caption_key", "image_hash"):
        seen = {}
        for i, value in enumerate(frame[column]):
            if value in seen:
                union(i, seen[value])
            else:
                seen[value] = i
    roots = [find(i) for i in range(len(frame))]
    return pd.factorize(np.asarray(roots, dtype=np.int64), sort=True)[0].astype(np.int64)

def assign_group_splits(frame, seed=42):
    groups = np.sort(frame["group_id"].unique())
    if len(groups) < 3:
        raise ValueError("Need at least three independent duplicate groups.")
    shuffled = np.random.default_rng(seed).permutation(groups)
    n_train = min(max(1, int(0.70 * len(groups))), len(groups) - 2)
    n_val = min(max(1, int(0.15 * len(groups))), len(groups) - n_train - 1)
    mapping = {int(g): ("train" if i < n_train else
                       "val" if i < n_train + n_val else "test")
               for i, g in enumerate(shuffled)}
    return frame["group_id"].map(mapping)

def prepare_catalog(dataset_root, max_products=None):
    dataset_root = Path(dataset_root)
    candidates = list(dataset_root.rglob("cleaned_products.csv"))
    candidates += list(dataset_root.rglob("styles.csv"))
    if not candidates:
        raise FileNotFoundError("Dataset folder needs styles.csv or cleaned_products.csv.")
    metadata_path = sorted(candidates, key=lambda p: (p.name != "cleaned_products.csv", str(p)))[0]
    audit = []
    def malformed(fields):
        audit.append({"stage": "csv", "id": fields[0] if fields else "",
                      "reason": "malformed_csv_record", "detail": f"{len(fields)} fields"})
        return None
    frame = pd.read_csv(metadata_path, dtype=str, keep_default_na=False,
                        engine="python", on_bad_lines=malformed).fillna("")
    if "id" not in frame or "productDisplayName" not in frame:
        raise ValueError("Metadata must include id and productDisplayName.")
    files = sorted(p for p in dataset_root.rglob("*")
                   if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    paths = {}
    for path in files:
        # Prefer the original images/ directory over optional HD copies.
        key = path.stem
        if key not in paths or path.parent.name == "images":
            paths[key] = path
    rows, seen_ids = [], set()
    for record in tqdm(frame.to_dict("records"), desc="Validate catalog"):
        raw_id = str(record.get("id", "")).strip()
        reason, detail = None, ""
        if not raw_id.isdigit():
            reason = "invalid_product_id"
        elif int(raw_id) in seen_ids:
            reason = "duplicate_product_id"
        else:
            product_id = int(raw_id)
            image_path = paths.get(str(product_id))
            caption = build_caption(record)
            if not image_path:
                reason = "missing_image"
            elif not caption or not str(record.get("productDisplayName", "")).strip():
                reason = "missing_product_name"
            else:
                try:
                    pixel_hash = decoded_image_hash(image_path)
                except (OSError, ValueError, Image.DecompressionBombError) as exc:
                    reason, detail = "corrupt_image", str(exc)
                if reason is None:
                    seen_ids.add(product_id)
                    rows.append({**record, "id": product_id,
                                 "images_path": str(image_path), "caption": caption,
                                 "caption_key": normalize_caption(caption), "image_hash": pixel_hash})
        if reason:
            audit.append({"stage": "cleaning", "id": raw_id, "reason": reason, "detail": detail})
    if not rows:
        raise ValueError("No valid product images and descriptions were found.")
    catalog = pd.DataFrame(rows).sort_values("id").reset_index(drop=True)
    if max_products is not None and len(catalog) > max_products:
        catalog = catalog.sample(n=max_products, random_state=SPLIT_SEED).sort_values("id").reset_index(drop=True)
    if len(catalog) < 30:
        raise ValueError("Fewer than 30 valid products; check dataset configuration.")
    catalog["group_id"] = duplicate_groups(catalog)
    catalog["split"] = assign_group_splits(catalog, SPLIT_SEED)
    assert catalog.groupby("group_id")["split"].nunique().max() == 1
    assert catalog.groupby("caption_key")["split"].nunique().max() == 1
    assert catalog.groupby("image_hash")["split"].nunique().max() == 1
    assert catalog["id"].is_unique
    return catalog, pd.DataFrame(audit, columns=["stage", "id", "reason", "detail"]), len(frame)


In [ ]:
if DATASET_DIR is None:
    import kagglehub
    dataset_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
else:
    dataset_root = Path(DATASET_DIR)
if not dataset_root.is_dir():
    raise FileNotFoundError(f"Dataset folder does not exist: {dataset_root}")
catalog, cleaning_audit, parsed_rows = prepare_catalog(dataset_root, MAX_PRODUCTS)
manifest_columns = ["id", "caption", "caption_key", "image_hash", "group_id", "split"]
manifest_csv = catalog[manifest_columns].to_csv(index=False)
MANIFEST_HASH = hashlib.sha256(manifest_csv.encode()).hexdigest()
EXPERIMENT_DIR = OUTPUT_DIR / ("experiment_" + MANIFEST_HASH[:16])
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
catalog.to_csv(EXPERIMENT_DIR / "catalog_manifest.csv", index=False)
cleaning_audit.to_csv(EXPERIMENT_DIR / "cleaning_audit.csv", index=False)
print("Dataset root:", dataset_root)
print("Parsed records:", parsed_rows, "| retained:", len(catalog), "| discarded:", len(cleaning_audit))
print("Duplicate groups:", catalog.group_id.nunique(), "| fingerprint:", MANIFEST_HASH)
display(catalog.groupby("split").agg(products=("id", "size"), groups=("group_id", "nunique")))
display(cleaning_audit.reason.value_counts().rename("discarded_records").to_frame())
if MAX_PRODUCTS is not None:
    print("SUBSET RUN: these results are not the full-dataset experiment.")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
catalog.articleType.value_counts().head(12).sort_values().plot.barh(ax=axes[0], title="Top product types")
catalog.split.value_counts().plot.bar(ax=axes[1], title="Products per split", rot=0)
fig.tight_layout()
fig.savefig(EXPERIMENT_DIR / "dataset_distribution.png", dpi=160)
plt.show()


## 4. Load frozen CLIP and cache aligned embeddings

Both Transformer encoders and CLIP's original projection layers remain frozen.
The same processor/checkpoint is used for catalog items and later queries.
Cache identity includes the manifest, resolved model commit, and preprocessing
policy. Completed extraction batches are recorded so an interrupted Colab run
can resume. Reduce `EXTRACT_BATCH_SIZE` after a GPU out-of-memory error.


In [ ]:
clip_model = CLIPModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION).to(DEVICE).eval()
clip_model.requires_grad_(False)
RESOLVED_REVISION = getattr(clip_model.config, "_commit_hash", None)
if not RESOLVED_REVISION:
    raise RuntimeError("Could not resolve model commit; set MODEL_REVISION explicitly.")
processor = CLIPProcessor.from_pretrained(MODEL_ID, revision=RESOLVED_REVISION, use_fast=False)
base_parameter_count = sum(p.numel() for p in clip_model.parameters())
print("Frozen CLIP parameters:", f"{base_parameter_count:,}")
print("Resolved checkpoint commit:", RESOLVED_REVISION)
assert not any(p.requires_grad for p in clip_model.parameters())
token_lengths = [len(x) for x in processor.tokenizer(catalog.caption.tolist(), truncation=False)["input_ids"]]
print("Descriptions truncated beyond 77 tokens:", sum(n > 77 for n in token_lengths))

def extraction_context():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if DEVICE.type == "cuda" else nullcontext()

def normalize_clip_output(value):
    # Transformers 4.x get_*_features returns projected feature tensors.
    if not isinstance(value, torch.Tensor):
        raise TypeError("Expected CLIP projected features; check Transformers version.")
    return F.normalize(value.float(), dim=-1).cpu().numpy().astype(np.float32)

@torch.inference_mode()
def encode_texts(texts, batch_size=64):
    arrays = []
    for start in range(0, len(texts), batch_size):
        inputs = processor(text=list(texts[start:start + batch_size]), return_tensors="pt",
                           padding=True, truncation=True, max_length=77).to(DEVICE)
        with extraction_context():
            features = clip_model.get_text_features(**inputs)
        arrays.append(normalize_clip_output(features))
    return np.ascontiguousarray(np.concatenate(arrays), dtype=np.float32)

@torch.inference_mode()
def encode_images(images):
    images = [ImageOps.exif_transpose(image).convert("RGB") for image in images]
    inputs = processor(images=images, return_tensors="pt").to(DEVICE)
    with extraction_context():
        features = clip_model.get_image_features(**inputs)
    return normalize_clip_output(features)

def extract_catalog_features(frame):
    cache_config = {"manifest_hash": MANIFEST_HASH, "model_id": MODEL_ID,
                    "model_revision": RESOLVED_REVISION, "rows": len(frame), "dimension": 512,
                    "processor": "checkpoint-default-slow-rgb-exif-transpose",
                    "text_max_length": 77, "cuda_autocast_fp16": DEVICE.type == "cuda"}
    cache_hash = hashlib.sha256(json.dumps(cache_config, sort_keys=True).encode()).hexdigest()[:16]
    cache_dir = OUTPUT_DIR / "feature_cache" / cache_hash
    cache_dir.mkdir(parents=True, exist_ok=True)
    meta_path = cache_dir / "cache_config.json"
    progress_path = cache_dir / "progress.json"
    image_path, text_path = cache_dir / "base_image.npy", cache_dir / "base_text.npy"
    if meta_path.exists():
        if json.loads(meta_path.read_text()) != cache_config:
            raise ValueError("Cache configuration mismatch; use a new output folder.")
        if not image_path.exists() or not text_path.exists():
            raise ValueError("Incomplete cache files; remove this cache folder and rerun.")
        image_array = np.load(image_path, mmap_mode="r+")
        text_array = np.load(text_path, mmap_mode="r+")
        start_row = json.loads(progress_path.read_text())["next_row"] if progress_path.exists() else 0
    else:
        image_array = np.lib.format.open_memmap(image_path, mode="w+", dtype=np.float32, shape=(len(frame), 512))
        text_array = np.lib.format.open_memmap(text_path, mode="w+", dtype=np.float32, shape=(len(frame), 512))
        meta_path.write_text(json.dumps(cache_config, indent=2))
        start_row = 0
    assert image_array.shape == text_array.shape == (len(frame), 512)
    for start in tqdm(range(start_row, len(frame), EXTRACT_BATCH_SIZE), desc="Frozen CLIP extraction"):
        stop = min(start + EXTRACT_BATCH_SIZE, len(frame))
        images = []
        for path in frame.images_path.iloc[start:stop]:
            with Image.open(path) as image:
                images.append(ImageOps.exif_transpose(image).convert("RGB").copy())
        image_array[start:stop] = encode_images(images)
        text_array[start:stop] = encode_texts(frame.caption.iloc[start:stop].tolist(), EXTRACT_BATCH_SIZE)
        image_array.flush()
        text_array.flush()
        temp = progress_path.with_suffix(".tmp")
        temp.write_text(json.dumps({"next_row": stop}))
        temp.replace(progress_path)
    for array in (image_array, text_array):
        assert np.isfinite(array).all()
        assert np.allclose(np.linalg.norm(array, axis=1), 1.0, atol=2e-4)
    return np.array(image_array), np.array(text_array), cache_config

base_image_features, base_text_features, feature_cache_config = extract_catalog_features(catalog)
print("Cached feature shapes:", base_image_features.shape, base_text_features.shape)


## 5. Trainable model, multi-positive loss, and metric definitions

Two independent residual heads each contain 262,912 parameters. Together with
one learned temperature scalar, the model has **525,825 trainable parameters**.
Initial adapter outputs equal original CLIP features. Duplicate-group members
are positives, avoiding contradictory negative labels for identical captions.

Evaluation reports both Success@K (at least one positive) and Recall@K (fraction
of all positives), plus MRR@10 and nDCG@10. Validation selects checkpoints using
mean bidirectional MRR@10. No test scores are used during training.


In [ ]:
"""Trainable residual adapters for frozen, normalized CLIP features.

This module does not change the application's existing search pipeline.
The Colab notebook exports an identical copy alongside the trained weights.
"""
import math
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F


class ResidualAdapter(nn.Module):
    def __init__(self, embedding_dim=512, hidden_dim=256, dropout=0.1, residual_weight=0.2):
        super().__init__()
        self.residual_weight = float(residual_weight)
        self.network = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embedding_dim),
        )
        # Start at the original CLIP representation, rather than a random space.
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, features):
        features = F.normalize(features.float(), dim=-1)
        return F.normalize(features + self.residual_weight * self.network(features), dim=-1)


class FashionAdapters(nn.Module):
    def __init__(self, embedding_dim=512, hidden_dim=256, dropout=0.1,
                 residual_weight=0.2, initial_temperature=0.07):
        super().__init__()
        args = (embedding_dim, hidden_dim, dropout, residual_weight)
        self.image_adapter = ResidualAdapter(*args)
        self.text_adapter = ResidualAdapter(*args)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1.0 / initial_temperature)))

    def forward(self, image_features, text_features):
        return self.image_adapter(image_features), self.text_adapter(text_features)

    def similarity_logits(self, image_features, text_features):
        images, texts = self(image_features, text_features)
        return self.logit_scale.exp().clamp(max=100.0) * (images @ texts.T)


def multi_positive_contrastive_loss(logits, group_ids):
    """Symmetric log-softmax loss with a uniform target over equivalent pairs.

    Every image/text pair in the same duplicate group is positive. This avoids
    penalizing duplicate descriptions as negatives. Each row always includes
    its paired item, so there must be at least one positive per row.
    """
    if logits.ndim != 2 or logits.shape[0] != logits.shape[1]:
        raise ValueError("Expected square image/text logits for paired batches.")
    if len(group_ids) != logits.shape[0]:
        raise ValueError("There must be one group ID per image/text pair.")
    positives = group_ids[:, None].eq(group_ids[None, :]).to(logits.dtype)
    targets = positives / positives.sum(dim=1, keepdim=True)
    image_loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
    text_loss = -(targets.T * F.log_softmax(logits.T, dim=1)).sum(dim=1).mean()
    return 0.5 * (image_loss + text_loss)


def adapter_config(config):
    keys = ("embedding_dim", "hidden_dim", "dropout", "residual_weight", "initial_temperature")
    return {key: config[key] for key in keys}


def load_adapters(checkpoint_path, device="cpu"):
    checkpoint = torch.load(Path(checkpoint_path), map_location="cpu", weights_only=True)
    model = FashionAdapters(**adapter_config(checkpoint["config"]))
    model.load_state_dict(checkpoint["model_state"])
    return model.to(device).eval(), checkpoint


@torch.inference_mode()
def adapt_features(model, features, modality, batch_size=1024):
    if modality not in {"image", "text"}:
        raise ValueError("modality must be image or text")
    head = model.image_adapter if modality == "image" else model.text_adapter
    device = next(model.parameters()).device
    model.eval()
    arrays = []
    for start in range(0, len(features), batch_size):
        batch = torch.as_tensor(np.asarray(features[start:start + batch_size]),
                                dtype=torch.float32, device=device)
        arrays.append(head(batch).cpu().numpy())
    if not arrays:
        return np.empty((0, head.network[0].in_features), dtype=np.float32)
    return np.ascontiguousarray(np.concatenate(arrays), dtype=np.float32)


@torch.inference_mode()
def retrieval_metrics(queries, gallery, query_groups, gallery_groups,
                      device="cpu", batch_size=256, cutoffs=(1, 5, 10)):
    """Exact blocked cosine evaluation; gallery must contain a positive per query.

    Success@K measures whether at least one relevant item is retrieved.
    Recall@K measures the fraction of ALL relevant gallery items retrieved.
    MRR and nDCG are truncated at the largest requested cutoff.
    """
    if len(queries) == 0 or len(gallery) == 0:
        raise ValueError("Retrieval evaluation requires nonempty queries and gallery.")
    queries = np.asarray(queries, dtype=np.float32)
    gallery = np.asarray(gallery, dtype=np.float32)
    query_groups = np.asarray(query_groups)
    gallery_groups = np.asarray(gallery_groups)
    if len(queries) != len(query_groups) or len(gallery) != len(gallery_groups):
        raise ValueError("Feature rows and group IDs must have matching lengths.")
    vectors = F.normalize(torch.as_tensor(gallery, device=device), dim=-1)
    gallery_ids = torch.as_tensor(gallery_groups, device=device)
    max_k = min(max(cutoffs), len(gallery))
    sums = {f"Success@{k}": 0.0 for k in cutoffs}
    sums.update({f"Recall@{k}": 0.0 for k in cutoffs})
    sums[f"MRR@{max(cutoffs)}"] = 0.0
    sums[f"nDCG@{max(cutoffs)}"] = 0.0
    discounts = 1.0 / torch.log2(torch.arange(2, max_k + 2, device=device).float())
    for start in range(0, len(queries), batch_size):
        stop = start + batch_size
        query = F.normalize(torch.as_tensor(queries[start:stop], device=device), dim=-1)
        ids = torch.as_tensor(query_groups[start:stop], device=device)
        scores = query @ vectors.T
        # Stable ordering makes ties deterministic in catalog row order.
        indices = torch.argsort(scores, dim=1, descending=True, stable=True)[:, :max_k]
        relevant = ids[:, None].eq(gallery_ids[indices])
        totals = ids[:, None].eq(gallery_ids[None, :]).sum(dim=1)
        if torch.any(totals == 0):
            raise ValueError("Every evaluated query must have a relevant gallery item.")
        for k in cutoffs:
            hits = relevant[:, :min(k, max_k)].sum(dim=1)
            sums[f"Success@{k}"] += (hits > 0).float().sum().item()
            sums[f"Recall@{k}"] += (hits / totals).sum().item()
        positions = torch.arange(1, max_k + 1, device=device).float()
        reciprocal = torch.where(relevant, 1.0 / positions, 0.0).max(dim=1).values
        sums[f"MRR@{max(cutoffs)}"] += reciprocal.sum().item()
        dcg = (relevant.float() * discounts).sum(dim=1)
        ideal = discounts.cumsum(dim=0)[totals.clamp(max=max_k) - 1]
        sums[f"nDCG@{max(cutoffs)}"] += (dcg / ideal).sum().item()
    return {key: value / len(queries) for key, value in sums.items()}

ADAPTER_MODULE_SOURCE = '"""Trainable residual adapters for frozen, normalized CLIP features.\n\nThis module does not change the application\'s existing search pipeline.\nThe Colab notebook exports an identical copy alongside the trained weights.\n"""\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass ResidualAdapter(nn.Module):\n    def __init__(self, embedding_dim=512, hidden_dim=256, dropout=0.1, residual_weight=0.2):\n        super().__init__()\n        self.residual_weight = float(residual_weight)\n        self.network = nn.Sequential(\n            nn.Linear(embedding_dim, hidden_dim),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(hidden_dim, embedding_dim),\n        )\n        # Start at the original CLIP representation, rather than a random space.\n        nn.init.zeros_(self.network[-1].weight)\n        nn.init.zeros_(self.network[-1].bias)\n\n    def forward(self, features):\n        features = F.normalize(features.float(), dim=-1)\n        return F.normalize(features + self.residual_weight * self.network(features), dim=-1)\n\n\nclass FashionAdapters(nn.Module):\n    def __init__(self, embedding_dim=512, hidden_dim=256, dropout=0.1,\n                 residual_weight=0.2, initial_temperature=0.07):\n        super().__init__()\n        args = (embedding_dim, hidden_dim, dropout, residual_weight)\n        self.image_adapter = ResidualAdapter(*args)\n        self.text_adapter = ResidualAdapter(*args)\n        self.logit_scale = nn.Parameter(torch.tensor(math.log(1.0 / initial_temperature)))\n\n    def forward(self, image_features, text_features):\n        return self.image_adapter(image_features), self.text_adapter(text_features)\n\n    def similarity_logits(self, image_features, text_features):\n        images, texts = self(image_features, text_features)\n        return self.logit_scale.exp().clamp(max=100.0) * (images @ texts.T)\n\n\ndef multi_positive_contrastive_loss(logits, group_ids):\n    """Symmetric log-softmax loss with a uniform target over equivalent pairs.\n\n    Every image/text pair in the same duplicate group is positive. This avoids\n    penalizing duplicate descriptions as negatives. Each row always includes\n    its paired item, so there must be at least one positive per row.\n    """\n    if logits.ndim != 2 or logits.shape[0] != logits.shape[1]:\n        raise ValueError("Expected square image/text logits for paired batches.")\n    if len(group_ids) != logits.shape[0]:\n        raise ValueError("There must be one group ID per image/text pair.")\n    positives = group_ids[:, None].eq(group_ids[None, :]).to(logits.dtype)\n    targets = positives / positives.sum(dim=1, keepdim=True)\n    image_loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()\n    text_loss = -(targets.T * F.log_softmax(logits.T, dim=1)).sum(dim=1).mean()\n    return 0.5 * (image_loss + text_loss)\n\n\ndef adapter_config(config):\n    keys = ("embedding_dim", "hidden_dim", "dropout", "residual_weight", "initial_temperature")\n    return {key: config[key] for key in keys}\n\n\ndef load_adapters(checkpoint_path, device="cpu"):\n    checkpoint = torch.load(Path(checkpoint_path), map_location="cpu", weights_only=True)\n    model = FashionAdapters(**adapter_config(checkpoint["config"]))\n    model.load_state_dict(checkpoint["model_state"])\n    return model.to(device).eval(), checkpoint\n\n\n@torch.inference_mode()\ndef adapt_features(model, features, modality, batch_size=1024):\n    if modality not in {"image", "text"}:\n        raise ValueError("modality must be image or text")\n    head = model.image_adapter if modality == "image" else model.text_adapter\n    device = next(model.parameters()).device\n    model.eval()\n    arrays = []\n    for start in range(0, len(features), batch_size):\n        batch = torch.as_tensor(np.asarray(features[start:start + batch_size]),\n                                dtype=torch.float32, device=device)\n        arrays.append(head(batch).cpu().numpy())\n    if not arrays:\n        return np.empty((0, head.network[0].in_features), dtype=np.float32)\n    return np.ascontiguousarray(np.concatenate(arrays), dtype=np.float32)\n\n\n@torch.inference_mode()\ndef retrieval_metrics(queries, gallery, query_groups, gallery_groups,\n                      device="cpu", batch_size=256, cutoffs=(1, 5, 10)):\n    """Exact blocked cosine evaluation; gallery must contain a positive per query.\n\n    Success@K measures whether at least one relevant item is retrieved.\n    Recall@K measures the fraction of ALL relevant gallery items retrieved.\n    MRR and nDCG are truncated at the largest requested cutoff.\n    """\n    if len(queries) == 0 or len(gallery) == 0:\n        raise ValueError("Retrieval evaluation requires nonempty queries and gallery.")\n    queries = np.asarray(queries, dtype=np.float32)\n    gallery = np.asarray(gallery, dtype=np.float32)\n    query_groups = np.asarray(query_groups)\n    gallery_groups = np.asarray(gallery_groups)\n    if len(queries) != len(query_groups) or len(gallery) != len(gallery_groups):\n        raise ValueError("Feature rows and group IDs must have matching lengths.")\n    vectors = F.normalize(torch.as_tensor(gallery, device=device), dim=-1)\n    gallery_ids = torch.as_tensor(gallery_groups, device=device)\n    max_k = min(max(cutoffs), len(gallery))\n    sums = {f"Success@{k}": 0.0 for k in cutoffs}\n    sums.update({f"Recall@{k}": 0.0 for k in cutoffs})\n    sums[f"MRR@{max(cutoffs)}"] = 0.0\n    sums[f"nDCG@{max(cutoffs)}"] = 0.0\n    discounts = 1.0 / torch.log2(torch.arange(2, max_k + 2, device=device).float())\n    for start in range(0, len(queries), batch_size):\n        stop = start + batch_size\n        query = F.normalize(torch.as_tensor(queries[start:stop], device=device), dim=-1)\n        ids = torch.as_tensor(query_groups[start:stop], device=device)\n        scores = query @ vectors.T\n        # Stable ordering makes ties deterministic in catalog row order.\n        indices = torch.argsort(scores, dim=1, descending=True, stable=True)[:, :max_k]\n        relevant = ids[:, None].eq(gallery_ids[indices])\n        totals = ids[:, None].eq(gallery_ids[None, :]).sum(dim=1)\n        if torch.any(totals == 0):\n            raise ValueError("Every evaluated query must have a relevant gallery item.")\n        for k in cutoffs:\n            hits = relevant[:, :min(k, max_k)].sum(dim=1)\n            sums[f"Success@{k}"] += (hits > 0).float().sum().item()\n            sums[f"Recall@{k}"] += (hits / totals).sum().item()\n        positions = torch.arange(1, max_k + 1, device=device).float()\n        reciprocal = torch.where(relevant, 1.0 / positions, 0.0).max(dim=1).values\n        sums[f"MRR@{max(cutoffs)}"] += reciprocal.sum().item()\n        dcg = (relevant.float() * discounts).sum(dim=1)\n        ideal = discounts.cumsum(dim=0)[totals.clamp(max=max_k) - 1]\n        sums[f"nDCG@{max(cutoffs)}"] += (dcg / ideal).sum().item()\n    return {key: value / len(queries) for key, value in sums.items()}\n'


In [ ]:
TRAIN_CONFIG = {
    "embedding_dim": 512, "hidden_dim": HIDDEN_DIM, "dropout": DROPOUT,
    "residual_weight": RESIDUAL_WEIGHT, "initial_temperature": INITIAL_TEMPERATURE,
    "model_id": MODEL_ID, "model_revision": RESOLVED_REVISION,
    "manifest_hash": MANIFEST_HASH, "feature_cache": feature_cache_config,
    "batch_size": TRAIN_BATCH_SIZE, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
    "min_lr_multiplier": MIN_LR_MULTIPLIER, "patience": PATIENCE,
    "min_delta": MIN_DELTA, "grad_clip_norm": GRAD_CLIP_NORM,
    "selection_metric": "mean_bidirectional_validation_MRR@10", "split_seed": SPLIT_SEED,
}
probe_model = FashionAdapters(**adapter_config(TRAIN_CONFIG))
trainable_parameters = sum(p.numel() for p in probe_model.parameters() if p.requires_grad)
expected_parameters = 2 * (2 * 512 * HIDDEN_DIM + HIDDEN_DIM + 512) + 1
assert trainable_parameters == expected_parameters
if HIDDEN_DIM == 256:
    assert trainable_parameters == 525825
print("Trainable adapter parameters:", f"{trainable_parameters:,}")
print("Frozen encoder parameters:", f"{base_parameter_count:,}")
sample = F.normalize(torch.randn(6, 512), dim=-1)
probe_model.eval()
assert torch.allclose(probe_model.image_adapter(sample), sample, atol=1e-6)
labels = torch.arange(6)
logits = torch.randn(6, 6)
expected_loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
assert torch.allclose(multi_positive_contrastive_loss(logits, labels), expected_loss)
assert torch.isfinite(multi_positive_contrastive_loss(logits, torch.tensor([0, 0, 1, 2, 2, 3])))
print("Model identity initialization and loss correctness checks passed.")
del probe_model

split_indices = {name: np.flatnonzero(catalog.split.to_numpy() == name)
                 for name in ("train", "val", "test")}
group_ids = catalog.group_id.to_numpy(dtype=np.int64)
assert min(map(len, split_indices.values())) >= 2

def evaluate_bidirectional(model, indices):
    images = base_image_features[indices]
    texts = base_text_features[indices]
    if model is not None:
        images = adapt_features(model, images, "image")
        texts = adapt_features(model, texts, "text")
    groups = group_ids[indices]
    args = {"device": DEVICE, "batch_size": EVAL_BATCH_SIZE}
    t2i = retrieval_metrics(texts, images, groups, groups, **args)
    i2t = retrieval_metrics(images, texts, groups, groups, **args)
    score = 0.5 * (t2i["MRR@10"] + i2t["MRR@10"])
    return {"text_to_image": t2i, "image_to_text": i2t}, float(score)

baseline_validation, baseline_validation_score = evaluate_bidirectional(None, split_indices["val"])
print("Original CLIP validation mean MRR@10:", round(baseline_validation_score, 4))
display(pd.DataFrame(baseline_validation).T)


## 6. Training with three seeds, validation selection, and resume

AdamW trains only the heads and temperature. The encoder is never part of the
optimizer. Each seed gets a best checkpoint and a resumable last checkpoint.
One epoch of warmup precedes cosine learning-rate decay. Early stopping uses
validation only. Single-item final batches are merged into the previous batch
because contrastive training needs negatives.


In [ ]:
def save_torch_atomic(value, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temporary)
    temporary.replace(path)

def capture_rng():
    state = np.random.get_state()
    return {"python": random.getstate(),
            "numpy": [state[0], state[1].tolist(), state[2], state[3], state[4]],
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}

def restore_rng(state):
    random.setstate(state["python"])
    n = state["numpy"]
    np.random.set_state((n[0], np.asarray(n[1], dtype=np.uint32), n[2], n[3], n[4]))
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state["cuda"]:
        torch.cuda.set_rng_state_all(state["cuda"])

def batch_indices(order, batch_size):
    batches = list(order.split(batch_size))
    if len(batches) > 1 and len(batches[-1]) == 1:
        batches[-2] = torch.cat([batches[-2], batches[-1]])
        batches.pop()
    return batches

def train_one_seed(seed):
    set_seed(seed)
    run_dir = EXPERIMENT_DIR / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    best_path, last_path = run_dir / "best.pt", run_dir / "last.pt"
    model = FashionAdapters(**adapter_config(TRAIN_CONFIG)).to(DEVICE)
    decayed, nondecayed = [], []
    for name, parameter in model.named_parameters():
        (decayed if parameter.ndim > 1 else nondecayed).append(parameter)
    optimizer = torch.optim.AdamW([
        {"params": decayed, "weight_decay": WEIGHT_DECAY},
        {"params": nondecayed, "weight_decay": 0.0}], lr=LEARNING_RATE, betas=(0.9, 0.999))
    indices = split_indices["train"]
    image_tensor = torch.from_numpy(base_image_features[indices])
    text_tensor = torch.from_numpy(base_text_features[indices])
    groups_tensor = torch.from_numpy(group_ids[indices])
    n_batches = len(batch_indices(torch.arange(len(indices)), TRAIN_BATCH_SIZE))
    total_steps = EPOCHS * n_batches
    warmup_steps = min(WARMUP_EPOCHS * n_batches, max(0, total_steps - 1))
    def lr_multiplier(step):
        if warmup_steps and step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = min(1.0, max(0.0, (step - warmup_steps) / max(1, total_steps - warmup_steps)))
        return MIN_LR_MULTIPLIER + (1 - MIN_LR_MULTIPLIER) * 0.5 * (1 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_multiplier)
    history, start_epoch, stale_epochs = [], 0, 0
    best_score, best_epoch = -1.0, -1
    if RESUME_TRAINING and last_path.exists():
        saved = torch.load(last_path, map_location="cpu", weights_only=True)
        if saved["config"] != TRAIN_CONFIG or saved["seed"] != seed:
            raise ValueError("Resume configuration mismatch. Use a new output folder or set RESUME_TRAINING=False.")
        model.load_state_dict(saved["model_state"])
        optimizer.load_state_dict(saved["optimizer_state"])
        scheduler.load_state_dict(saved["scheduler_state"])
        history = saved["history"]
        start_epoch, stale_epochs = saved["epoch"] + 1, saved["stale_epochs"]
        best_score, best_epoch = saved["best_score"], saved["best_epoch"]
        restore_rng(saved["rng_state"])
        print(f"Seed {seed}: resume from epoch {start_epoch + 1}")
        if not best_path.exists():
            raise FileNotFoundError("Resume checkpoint exists without best.pt; restore both checkpoints.")
    for epoch in range(start_epoch, EPOCHS):
        if stale_epochs >= PATIENCE:
            break
        started = time.perf_counter()
        model.train()
        order = torch.randperm(len(indices))
        loss_sum, samples = 0.0, 0
        for batch in tqdm(batch_indices(order, TRAIN_BATCH_SIZE),
                          desc=f"Seed {seed}, epoch {epoch + 1}", leave=False):
            images = image_tensor[batch].to(DEVICE)
            texts = text_tensor[batch].to(DEVICE)
            groups = groups_tensor[batch].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model.similarity_logits(images, texts)
            loss = multi_positive_contrastive_loss(logits, groups)
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite training loss; check features and learning rate.")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
            with torch.no_grad():
                model.logit_scale.clamp_(0, math.log(100))
            scheduler.step()
            loss_sum += loss.item() * len(batch)
            samples += len(batch)
        model.eval()
        validation, selection_score = evaluate_bidirectional(model, split_indices["val"])
        improved = selection_score > best_score + MIN_DELTA
        if improved:
            best_score, best_epoch, stale_epochs = selection_score, epoch, 0
            save_torch_atomic({"config": TRAIN_CONFIG, "seed": seed, "epoch": epoch,
                               "best_validation_score": best_score,
                               "model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                               "validation_metrics": validation}, best_path)
        else:
            stale_epochs += 1
        row = {"epoch": epoch + 1, "train_loss": loss_sum / samples,
               "validation_score": selection_score, "learning_rate": optimizer.param_groups[0]["lr"],
               "temperature": float(model.logit_scale.detach().exp().reciprocal().cpu()),
               "epoch_seconds": time.perf_counter() - started, "improved": improved}
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        save_torch_atomic({"config": TRAIN_CONFIG, "seed": seed, "epoch": epoch,
                           "best_score": best_score, "best_epoch": best_epoch, "stale_epochs": stale_epochs,
                           "model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                           "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
                           "history": history, "rng_state": capture_rng()}, last_path)
        print(f"Seed {seed} epoch {epoch + 1}: loss={row['train_loss']:.4f}, "
              f"val MRR@10={selection_score:.4f}, best={best_score:.4f}")
    best_model, checkpoint = load_adapters(best_path, DEVICE)
    assert checkpoint["config"]["manifest_hash"] == MANIFEST_HASH
    return {"seed": seed, "checkpoint": str(best_path),
            "best_validation_score": float(checkpoint["best_validation_score"]),
            "best_epoch": int(checkpoint["epoch"]) + 1, "history": history}


In [ ]:
seed_runs = [train_one_seed(seed) for seed in TRAIN_SEEDS]
selection_table = pd.DataFrame([{k: v for k, v in run.items() if k != "history"} for run in seed_runs])
selection_table.to_csv(EXPERIMENT_DIR / "validation_selection.csv", index=False)
display(selection_table)
chosen_run = max(seed_runs, key=lambda run: run["best_validation_score"])
deployment_model, deployment_checkpoint = load_adapters(chosen_run["checkpoint"], DEVICE)
print("Deployment seed selected using validation ONLY:", chosen_run["seed"])
print("Training is complete. Test evaluation starts in the next section.")


## 7. Final held-out test evaluation and plots

Test descriptions query the test image gallery and vice versa. All variants use
identical relevance labels and gallery rows. Duplicate-group positives may be
multiple products, so Success and Recall have distinct meanings. Report gallery
size alongside scores. The baseline is deterministic for cached features; mean
and sample standard deviation describe the three adapted training seeds.


In [ ]:
metric_rows = []
def add_metric_rows(variant, seed, metrics):
    for direction, values in metrics.items():
        metric_rows.append({"variant": variant, "seed": seed, "direction": direction,
                            "gallery_size": len(split_indices["test"]),
                            "query_count": len(split_indices["test"]), **values})

baseline_test, _ = evaluate_bidirectional(None, split_indices["test"])
add_metric_rows("original_clip", -1, baseline_test)
for run in seed_runs:
    evaluated_model, _ = load_adapters(run["checkpoint"], DEVICE)
    values, _ = evaluate_bidirectional(evaluated_model, split_indices["test"])
    add_metric_rows("trained_residual_heads", run["seed"], values)
    del evaluated_model
results = pd.DataFrame(metric_rows)
results.to_csv(EXPERIMENT_DIR / "test_metrics_by_seed.csv", index=False)
metric_names = [column for column in results if "@" in column]
summary = results.groupby(["variant", "direction"])[metric_names].agg(["mean", "std"])
summary.to_csv(EXPERIMENT_DIR / "test_metrics_summary.csv")
print("Test gallery products:", len(split_indices["test"]))
display(results)
display(summary)
print("Baseline std is NaN because it has one deterministic run, not repeated training.")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for run in seed_runs:
    history = pd.DataFrame(run["history"])
    for ax, column, title in zip(axes.flat,
        ["train_loss", "validation_score", "learning_rate", "temperature"],
        ["Training contrastive loss", "Validation mean MRR@10", "Learning rate", "Learned temperature"]):
        ax.plot(history.epoch, history[column], label=f"seed {run['seed']}")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend()
fig.tight_layout()
fig.savefig(EXPERIMENT_DIR / "training_curves.png", dpi=180)
plt.show()

comparison = results.groupby(["variant", "direction"])[["Success@1", "Success@5", "Success@10"]].mean()
ax = comparison.plot.bar(figsize=(11, 4), ylim=(0, 1), rot=15, title="Held-out retrieval: mean success across seeds")
ax.set_ylabel("Fraction of queries with a relevant result")
ax.figure.tight_layout()
ax.figure.savefig(EXPERIMENT_DIR / "test_retrieval_comparison.png", dpi=180)
plt.show()


## 8. Optional short metadata-query and reranking experiment

This experiment uses synthetic queries such as “Men Red Sports Shoes”. A result
is relevant when its article type, colour, and audience all agree with the query.
This is a different relevance definition from paired-caption retrieval. The
metadata reranker sees catalog attributes: it is an explicitly disclosed heuristic,
not another neural model or evidence of human-query performance. It reranks the
top 100 adapted candidates with a +0.2 boost per exact attribute match.

The trained variant here is the validation-selected deployment seed. It is not
an average across three seeds and it does not select the checkpoint.


In [ ]:
def metadata_reranked_metrics(query_features, image_features, query_keys, gallery_keys,
                             query_attributes, gallery_attributes):
    metric_sums = {f"{metric}@{k}": 0.0 for metric in ("Success", "Recall") for k in (1, 5, 10)}
    metric_sums.update({"MRR@10": 0.0, "nDCG@10": 0.0})
    gallery_tensor = torch.as_tensor(image_features, dtype=torch.float32, device=DEVICE)
    gallery_ids_tensor = torch.as_tensor(gallery_keys, device=DEVICE)
    for start in range(0, len(query_features), EVAL_BATCH_SIZE):
        stop = start + EVAL_BATCH_SIZE
        queries = torch.as_tensor(query_features[start:stop], dtype=torch.float32, device=DEVICE)
        scores = queries @ gallery_tensor.T
        candidates = torch.argsort(scores, descending=True, stable=True)[:, :min(100, len(image_features))]
        boosts = np.zeros((len(queries), len(image_features)), dtype=np.float32)
        for column in range(3):
            boosts += 0.2 * (query_attributes[start:stop, column, None] == gallery_attributes[None, :, column])
        boosted = scores + torch.as_tensor(boosts, device=DEVICE)
        masked = torch.full_like(boosted, -torch.inf)
        masked.scatter_(1, candidates, boosted.gather(1, candidates))
        top_k = min(10, len(image_features))
        ranks = torch.argsort(masked, descending=True, stable=True)[:, :top_k]
        ids = torch.as_tensor(query_keys[start:stop], device=DEVICE)
        relevance = ids[:, None].eq(gallery_ids_tensor[ranks])
        totals = ids[:, None].eq(gallery_ids_tensor[None, :]).sum(1)
        assert torch.all(totals > 0)
        for k in (1, 5, 10):
            hits = relevance[:, :min(k, top_k)].sum(1)
            metric_sums[f"Success@{k}"] += (hits > 0).float().sum().item()
            metric_sums[f"Recall@{k}"] += (hits / totals).sum().item()
        positions = torch.arange(1, top_k + 1, device=DEVICE).float()
        metric_sums["MRR@10"] += torch.where(relevance, 1 / positions, 0).max(1).values.sum().item()
        discounts = 1 / torch.log2(positions + 1)
        ideal = discounts.cumsum(0)[totals.clamp(max=top_k) - 1]
        metric_sums["nDCG@10"] += ((relevance * discounts).sum(1) / ideal).sum().item()
    return {key: value / len(query_features) for key, value in metric_sums.items()}

metadata_query_results = None
if RUN_METADATA_QUERY_EXPERIMENT:
    test_rows = catalog.iloc[split_indices["test"]].copy()
    attribute_columns = ["articleType", "baseColour", "gender"]
    complete = test_rows[attribute_columns].apply(lambda col: col.astype(str).str.strip().ne("")).all(axis=1)
    unique_queries = test_rows.loc[complete, attribute_columns].drop_duplicates().sort_values(attribute_columns)
    if len(unique_queries) > METADATA_QUERY_LIMIT:
        unique_queries = unique_queries.sample(METADATA_QUERY_LIMIT, random_state=SPLIT_SEED).sort_values(attribute_columns)
    if len(unique_queries):
        query_attributes = unique_queries.apply(lambda col: col.astype(str).str.strip().str.casefold()).to_numpy()
        gallery_attributes = test_rows[attribute_columns].apply(lambda col: col.astype(str).str.strip().str.casefold()).to_numpy()
        gallery_tuples = [tuple(row) for row in gallery_attributes]
        key_mapping = {value: i for i, value in enumerate(sorted(set(gallery_tuples)))}
        gallery_keys = np.asarray([key_mapping[value] for value in gallery_tuples], dtype=np.int64)
        query_keys = np.asarray([key_mapping[tuple(value)] for value in query_attributes], dtype=np.int64)
        query_texts = [f"{row.gender} {row.baseColour} {row.articleType}" for row in unique_queries.itertuples()]
        query_base = encode_texts(query_texts)
        query_adapted = adapt_features(deployment_model, query_base, "text")
        gallery_base = base_image_features[split_indices["test"]]
        gallery_adapted = adapt_features(deployment_model, gallery_base, "image")
        args = dict(query_groups=query_keys, gallery_groups=gallery_keys,
                    device=DEVICE, batch_size=EVAL_BATCH_SIZE)
        records = []
        for variant, queries, images in (("original_clip", query_base, gallery_base),
                                        ("trained_residual_heads", query_adapted, gallery_adapted)):
            records.append({"variant": variant, "query_count": len(query_texts),
                            "gallery_size": len(test_rows), **retrieval_metrics(queries, images, **args)})
        records.append({"variant": "trained_heads_plus_metadata_reranking",
                        "query_count": len(query_texts), "gallery_size": len(test_rows),
                        **metadata_reranked_metrics(query_adapted, gallery_adapted, query_keys,
                                                   gallery_keys, query_attributes, gallery_attributes)})
        metadata_query_results = pd.DataFrame(records)
        metadata_query_results["selected_training_seed"] = chosen_run["seed"]
        metadata_query_results.to_csv(EXPERIMENT_DIR / "synthetic_metadata_query_metrics.csv", index=False)
        pd.DataFrame({"query": query_texts, "relevance_key": query_keys}).to_csv(
            EXPERIMENT_DIR / "synthetic_metadata_queries.csv", index=False)
        display(metadata_query_results)
    else:
        print("No complete attribute tuples; metadata-query experiment skipped.")


## 9. Build deployment indexes and try text, image, and combined retrieval

The demo indexes the complete catalog, including training products, to demonstrate
the application. Its gallery is deliberately different from the held-out test
gallery; demo outputs are not test results. A text query directly searches image
embeddings. A combined query averages the adapted text/image vectors and then
normalizes the result; alpha is an inference setting, not a trained fusion layer.


In [ ]:
adapted_image_features = adapt_features(deployment_model, base_image_features, "image")
adapted_text_features = adapt_features(deployment_model, base_text_features, "text")
assert adapted_image_features.shape == adapted_text_features.shape == (len(catalog), 512)
assert np.allclose(np.linalg.norm(adapted_image_features, axis=1), 1, atol=2e-4)
assert np.allclose(np.linalg.norm(adapted_text_features, axis=1), 1, atol=2e-4)
image_index = faiss.IndexFlatIP(512)
text_index = faiss.IndexFlatIP(512)
image_index.add(adapted_image_features)
text_index.add(adapted_text_features)
assert image_index.ntotal == text_index.ntotal == len(catalog)

def search_products(query_text=None, query_image=None, alpha=0.5, top_k=10, exclude_product_id=None):
    if not 0 <= alpha <= 1:
        raise ValueError("alpha must be between 0 and 1")
    vectors, weights = [], []
    if query_text and query_text.strip():
        vectors.append(adapt_features(deployment_model, encode_texts([query_text]), "text")[0])
        weights.append(alpha if query_image is not None else 1.0)
    if query_image is not None:
        vectors.append(adapt_features(deployment_model, encode_images([query_image]), "image")[0])
        weights.append(1 - alpha if query_text and query_text.strip() else 1.0)
    if not vectors:
        raise ValueError("Supply text, an image, or both")
    query = sum(weight * vector for weight, vector in zip(weights, vectors))
    norm = np.linalg.norm(query)
    if norm < 1e-8:
        raise ValueError("Combined query has near-zero norm; change alpha or inputs")
    query = np.ascontiguousarray((query / norm).reshape(1, -1), dtype=np.float32)
    count = min(len(catalog), top_k + int(exclude_product_id is not None))
    scores, row_ids = image_index.search(query, count)
    records = []
    for row_id, score in zip(row_ids[0], scores[0]):
        if row_id < 0:
            continue
        row = catalog.iloc[int(row_id)]
        if exclude_product_id is not None and int(row.id) == int(exclude_product_id):
            continue
        records.append({"row_id": int(row_id), "id": int(row.id),
                        "name": row.productDisplayName, "score": float(score), "image_path": row.images_path})
        if len(records) == top_k:
            break
    return pd.DataFrame(records)

def show_products(matches, title="Retrieved products"):
    if matches.empty:
        print("No results")
        return
    columns = 5
    rows = math.ceil(len(matches) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(15, 3.5 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, row in zip(axes.flat, matches.itertuples()):
        with Image.open(row.image_path) as image:
            ax.imshow(ImageOps.exif_transpose(image).convert("RGB"))
        ax.set_title(f"{row.name[:48]}\nID {row.id} | cosine {row.score:.3f}", fontsize=9)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


In [ ]:
# Edit these values and rerun this cell to explore retrieval.
QUERY_TEXT = "red sports shoes for men"
QUERY_IMAGE_PATH = None  # e.g. "/content/my_shoe_photo.jpg"
UPLOAD_QUERY_IMAGE = False  # set True to choose an image from your computer
TEXT_WEIGHT = 0.5

if UPLOAD_QUERY_IMAGE:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        QUERY_IMAGE_PATH = next(iter(uploaded))
query_image = None
if QUERY_IMAGE_PATH:
    with Image.open(QUERY_IMAGE_PATH) as image:
        query_image = ImageOps.exif_transpose(image).convert("RGB").copy()
matches = search_products(QUERY_TEXT, query_image, alpha=TEXT_WEIGHT)
display(matches[["id", "name", "score"]])
show_products(matches, "Text/image query retrieval")

# Catalog-image demonstration: exclude the source product from its own results.
source_product = catalog.iloc[split_indices["test"][0]]
with Image.open(source_product.images_path) as image:
    example_image = image.convert("RGB").copy()
image_matches = search_products(query_image=example_image, exclude_product_id=int(source_product.id))
show_products(image_matches, f"Similar products to ID {source_product.id}; source excluded")


## 10. Measure latency without model loading or display

Warm up first, then measure repeated queries. Synchronize CUDA before/after
timing. Search-only latency is not end-to-end latency. The measurements describe
this Colab runtime and catalog size, not guaranteed deployment performance.


In [ ]:
def synchronize_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def time_function(function, repeats=20, warmups=3):
    for _ in range(warmups):
        function()
    milliseconds = []
    for _ in range(repeats):
        synchronize_device()
        started = time.perf_counter()
        function()
        synchronize_device()
        milliseconds.append((time.perf_counter() - started) * 1000)
    return {"mean_ms": float(np.mean(milliseconds)), "median_ms": float(np.median(milliseconds)),
            "p95_ms": float(np.percentile(milliseconds, 95)), "repeats": repeats}

timing_text = "blue shirt for men"
timing_base = encode_texts([timing_text])
timing_adapted = adapt_features(deployment_model, timing_base, "text")
timing_functions = {
    "clip_text_encoding": lambda: encode_texts([timing_text]),
    "text_adapter": lambda: adapt_features(deployment_model, timing_base, "text"),
    "faiss_image_search_only": lambda: image_index.search(timing_adapted, 10),
    "text_query_end_to_end": lambda: image_index.search(
        adapt_features(deployment_model, encode_texts([timing_text]), "text"), 10),
}
latencies = pd.DataFrame([{"stage": stage, "catalog_size": len(catalog), "device": str(DEVICE),
                            **time_function(function)} for stage, function in timing_functions.items()])
latencies.to_csv(EXPERIMENT_DIR / "latency.csv", index=False)
display(latencies)


## 11. Save deployment bundle and verify reload

The deployment ZIP contains the learned heads, their Python definitions, the
base checkpoint identity, aligned metadata, both FAISS indexes, reports, figures,
and instructions. Base CLIP weights are downloaded separately at the recorded
revision. Large baseline feature caches and resumable optimizer checkpoints
remain in Drive. Adapted arrays are saved outside the ZIP to avoid duplicating
the same vectors already contained in the indexes.

The existing application needs adapter loading and query embedding changes before
these files can be used. Never mix old indexes with newly adapted query vectors.


In [ ]:
DEPLOY_DIR = EXPERIMENT_DIR / "deployment"
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(chosen_run["checkpoint"], DEPLOY_DIR / "fashion_adapters.pt")
(DEPLOY_DIR / "fashion_adapters.py").write_text(ADAPTER_MODULE_SOURCE, encoding="utf-8")
export_config = {**TRAIN_CONFIG, "selected_seed": chosen_run["seed"],
                 "best_epoch": chosen_run["best_epoch"],
                 "best_validation_score": chosen_run["best_validation_score"],
                 "base_parameter_count": base_parameter_count,
                 "trainable_parameter_count": trainable_parameters,
                 "catalog_size": len(catalog), "train_seeds": TRAIN_SEEDS,
                 "text_search_index": "image_index.faiss", "image_search_index": "image_index.faiss",
                 "text_index_purpose": "image-to-text or text-to-text retrieval",
                 "fusion": {"method": "normalized_weighted_sum", "default_alpha": 0.5},
                 "test_gallery_size": len(split_indices["test"]),
                 "relevance": "same_duplicate_group_for_primary_paired_retrieval"}
(DEPLOY_DIR / "model_config.json").write_text(json.dumps(export_config, indent=2), encoding="utf-8")
export_catalog = catalog.copy()
export_catalog["index_row"] = np.arange(len(catalog))
export_catalog["image_filename"] = export_catalog.images_path.map(lambda value: Path(value).name)
export_catalog.to_csv(DEPLOY_DIR / "catalog_metadata.csv", index=False)
catalog[manifest_columns].to_csv(DEPLOY_DIR / "split_manifest.csv", index=False)
faiss.write_index(image_index, str(DEPLOY_DIR / "image_index.faiss"))
faiss.write_index(text_index, str(DEPLOY_DIR / "text_index.faiss"))
np.save(EXPERIMENT_DIR / "adapted_image_features.npy", adapted_image_features)
np.save(EXPERIMENT_DIR / "adapted_text_features.npy", adapted_text_features)
for report in EXPERIMENT_DIR.iterdir():
    if report.is_file() and report.suffix in {".csv", ".json", ".png"}:
        shutil.copy2(report, DEPLOY_DIR / report.name)
shutil.copy2(OUTPUT_DIR / "requirements_freeze.txt", DEPLOY_DIR / "requirements_freeze.txt")
for run in seed_runs:
    history = Path(run["checkpoint"]).parent / "history.csv"
    shutil.copy2(history, DEPLOY_DIR / f"training_history_seed_{run['seed']}.csv")

readme = f"""# Fashion CLIP deployment bundle

Base model: {MODEL_ID}
Exact Hugging Face revision: {RESOLVED_REVISION}
Trainable parameters: {trainable_parameters:,}
Selected training seed: {chosen_run['seed']} (validation only)
Catalog rows: {len(catalog)}
Primary test gallery rows: {len(split_indices['test'])}
Manifest SHA256: {MANIFEST_HASH}

This trains residual projection heads on frozen CLIP features. It does not
fine-tune the CLIP Transformer encoders. No improvement is assumed.

## Loading

Put fashion_adapters.py on the Python import path. Load the checkpoint with
load_adapters('fashion_adapters.pt', device). Load CLIPModel and CLIPProcessor
from the model ID AND exact revision above. Create normalized float32 base
features, apply the matching text/image adapter, and normalize the result.

## Search and row alignment

Text queries search image_index.faiss using adapted TEXT embeddings.
Image queries search image_index.faiss using adapted IMAGE embeddings.
text_index.faiss contains adapted product-caption embeddings for reverse
image-to-text or optional text-to-text retrieval.
Index row i maps to catalog_metadata.csv row i. Do not sort or filter the
metadata after loading without rebuilding the indexes in the same order.
Colab images_path entries are nonportable: resolve local thumbnails by id.
Never use adapted query features with the old application's original indexes.
Update the application's embedding functions and route text-to-image search
to the image index before installing this bundle.

## Combined query

Normalize alpha * adapted_text + (1-alpha) * adapted_image, then search the
image index. Default alpha=0.5; this is an inference heuristic, not a trained
fusion module. Exclude the source item when demonstrating catalog-image search.

## Reports

test_metrics_by_seed.csv: paired-caption held-out metrics, original vs adapted.
test_metrics_summary.csv: means and sample standard deviations.
synthetic_metadata_query_metrics.csv, if present: separate synthetic short-query
experiment; metadata reranking accesses known catalog attributes.
latency.csv: encoding/adapter/search timings exclude model loading and display.
training_history_seed_*.csv and PNGs: actual training histories and figures.
split_manifest.csv and cleaning_audit.csv: split and cleaning provenance.
requirements_freeze.txt: versions from this runtime; not a minimal installer.

## Limitations

Small thumbnails, synthetic descriptions, ambiguous captions, annotation noise,
and undetected near duplicates limit generalization claims. Test galleries here
contain held-out products only; deployment searches the complete catalog.
No novel-architecture or state-of-the-art claim is made.
"""
(DEPLOY_DIR / "README.md").write_text(readme, encoding="utf-8")

reloaded_model, _ = load_adapters(DEPLOY_DIR / "fashion_adapters.pt", DEVICE)
reloaded_index = faiss.read_index(str(DEPLOY_DIR / "image_index.faiss"))
reloaded_metadata = pd.read_csv(DEPLOY_DIR / "catalog_metadata.csv")
assert reloaded_index.ntotal == len(reloaded_metadata) == len(catalog)
assert reloaded_metadata.id.tolist() == catalog.id.tolist()
reencoded = adapt_features(reloaded_model, base_text_features[:4], "text")
assert np.allclose(reencoded, adapted_text_features[:4], atol=1e-6)
assert np.allclose(reloaded_index.reconstruct(0), adapted_image_features[0], atol=1e-6)
print("Checkpoint reload and index/metadata row-alignment checks passed.")

ZIP_PATH = EXPERIMENT_DIR / "fashion_clip_deployment.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(DEPLOY_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(DEPLOY_DIR)))
print("Deployment ZIP:", ZIP_PATH)
print("ZIP size MB:", round(ZIP_PATH.stat().st_size / 2**20, 1))
print("All experiment outputs:", EXPERIMENT_DIR)


In [ ]:
# Optional browser download; the same ZIP is already saved in Drive.
DOWNLOAD_ZIP_NOW = False
if DOWNLOAD_ZIP_NOW:
    from google.colab import files
    files.download(str(ZIP_PATH))


## 12. Report and viva checklist

Explain the research question, dataset and cleaning, duplicate-group split,
CLIP image/text Transformers, frozen parameters versus trainable parameters,
residual adapter equation, in-batch negatives and duplicate positives, loss,
optimizer/schedule, validation selection, and independent final test evaluation.

Include the measured baseline and adapted results, means/std across training
seeds, loss/validation curves, qualitative successes and failures, and separate
search-only/end-to-end timings. A negative result remains a valid finding.

Suggested claim: **“We evaluated lightweight residual-head adaptation of frozen
CLIP features for multimodal fashion retrieval.”** Only say performance improved
if the recorded held-out results support it. The metadata-query experiment is
synthetic and uses a different relevance definition; disclose this in tables.

**Troubleshooting:** Reduce extraction batch for GPU OOM. For interrupted runs,
rerun cells in order with the same settings/Drive folder; feature extraction and
training resume. Changing training settings requires a new output folder or
`RESUME_TRAINING=False`. Do not change cached feature files manually. If Kaggle
requests authentication, authenticate or use `DATASET_DIR`. A subset smoke run
and full run get different manifest folders and caches.

**Extensions:** Actual encoder fine-tuning/LoRA, higher-resolution images, human
query relevance labels, and hard negatives are separate follow-up experiments.


## Preparation checks

Before delivery, every code cell was parsed as Python, and the actual notebook
definitions were exercised locally on small inputs. Checks covered trainable
parameter counts, identity initialization, multi-positive loss and gradients,
metric definitions, image cleaning, transitive duplicate grouping and split
isolation, training updates, deterministic interrupted-training resume,
checkpoint/FAISS reload, text/image/fused query logic and source-item exclusion,
metadata reranking, and CLIP image/text feature extraction/cache reuse on two
real project products.

These checks do not constitute full-dataset results. The dependency installation,
Kaggle download, Drive integration, complete experiment, figures, and GPU execution
must run in your Colab session. This notebook intentionally contains no fabricated
training outputs or performance numbers.
